
# 4060 Environment Check
## Torch / CUDA / HF / harness sanity check

Run these cells top to bottom.

This notebook checks:
- active Python / conda environment
- PyTorch and CUDA visibility
- GPU name and VRAM
- `transformers`, `accelerate`, `sentence_transformers`
- optional HF token presence
- tiny live load test for:
  - generation model
  - embedding model

Use this before rerunning the main harness.


In [1]:

import os
import sys
import platform
import importlib

print("Python executable:", sys.executable)
print("Python version:", sys.version)
print("Platform:", platform.platform())
print("Conda prefix:", os.environ.get("CONDA_PREFIX", "(not set)"))
print("Conda env:", os.environ.get("CONDA_DEFAULT_ENV", "(not set)"))


Python executable: C:\Users\Developer\anaconda3\envs\nexus-ultimate\python.exe
Python version: 3.11.14 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 18:30:03) [MSC v.1929 64 bit (AMD64)]
Platform: Windows-10-10.0.19045-SP0
Conda prefix: C:\Users\Developer\anaconda3\envs\nexus-ultimate
Conda env: nexus-ultimate


In [2]:

def pkg_version(name: str):
    try:
        mod = importlib.import_module(name)
        return getattr(mod, "__version__", "(no __version__)")
    except Exception as e:
        return f"NOT INSTALLED: {e}"

for name in [
    "torch",
    "transformers",
    "accelerate",
    "sentence_transformers",
    "huggingface_hub",
    "numpy",
    "pandas",
    "matplotlib",
    "tqdm",
]:
    print(f"{name}: {pkg_version(name)}")


torch: 2.11.0+cpu
transformers: 5.3.0
accelerate: 1.13.0
sentence_transformers: 5.4.1
huggingface_hub: 1.7.2
numpy: 2.4.3
pandas: 3.0.0
matplotlib: 3.10.8
tqdm: 4.67.3


In [3]:

import torch

print("torch version:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("cuda device count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("current device:", torch.cuda.current_device())
    print("gpu name:", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print("total VRAM (GB):", round(props.total_memory / (1024**3), 2))
    print("capability:", f"{props.major}.{props.minor}")
    print("cuda runtime:", torch.version.cuda)
else:
    print("No CUDA device visible to torch.")


torch version: 2.11.0+cpu
cuda available: False
cuda device count: 0
No CUDA device visible to torch.


In [4]:

if torch.cuda.is_available():
    x = torch.randn(2048, 2048, device="cuda")
    y = torch.randn(2048, 2048, device="cuda")
    z = x @ y
    torch.cuda.synchronize()
    print("GPU matmul test: OK", z.shape, z.dtype, z.device)
else:
    print("Skipped GPU matmul test because CUDA is unavailable.")


Skipped GPU matmul test because CUDA is unavailable.


In [5]:

HF_TOKEN = os.getenv("HF_TOKEN", "")
print("HF token present:", bool(HF_TOKEN))
print("HF_HUB_DISABLE_SYMLINKS_WARNING:", os.getenv("HF_HUB_DISABLE_SYMLINKS_WARNING", "(not set)"))


HF token present: False



## Tiny live load test

Start with the same small pair used for the harness.
If the models are already cached, set `LOCAL_FILES_ONLY = True`.


In [6]:

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
LOCAL_FILES_ONLY = False

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

print("MODEL_NAME =", MODEL_NAME)
print("EMBED_MODEL_NAME =", EMBED_MODEL_NAME)
print("LOCAL_FILES_ONLY =", LOCAL_FILES_ONLY)
print("DEVICE =", DEVICE)
print("DTYPE =", DTYPE)


MODEL_NAME = Qwen/Qwen2.5-0.5B-Instruct
EMBED_MODEL_NAME = sentence-transformers/all-MiniLM-L6-v2
LOCAL_FILES_ONLY = False
DEVICE = cpu
DTYPE = torch.float32


In [7]:

from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModel

hf_kwargs = {"local_files_only": LOCAL_FILES_ONLY}
if HF_TOKEN:
    hf_kwargs["token"] = HF_TOKEN

print("Loading tokenizer...")
tok = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True, **hf_kwargs)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

print("Loading generation model...")
lm = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=DTYPE,
    device_map="auto" if DEVICE == "cuda" else None,
    **hf_kwargs,
)
if DEVICE != "cuda":
    lm = lm.to(DEVICE)
lm.eval()

print("Generation model loaded OK.")
print("Model class:", type(lm).__name__)


Loading tokenizer...


Loading generation model...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Generation model loaded OK.
Model class: Qwen2ForCausalLM


In [8]:

prompt = "Question: What is the capital of Japan?\nChoices:\nA. Beijing\nB. Seoul\nC. Tokyo\nD. Kyoto\n\nAnswer with the single correct letter only."
inputs = tok(prompt, return_tensors="pt")
inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

with torch.no_grad():
    out = lm.generate(
        **inputs,
        max_new_tokens=4,
        do_sample=False,
        pad_token_id=tok.pad_token_id,
    )

decoded = tok.decode(out[0], skip_special_tokens=True)
print(decoded)


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Question: What is the capital of Japan?
Choices:
A. Beijing
B. Seoul
C. Tokyo
D. Kyoto

Answer with the single correct letter only. C


In [9]:

try:
    from sentence_transformers import SentenceTransformer

    print("Loading sentence-transformers embedding model...")
    emb = SentenceTransformer(EMBED_MODEL_NAME, device=DEVICE)
    vec = emb.encode(
        ["What is the capital of Japan?", "Tokyo is the capital of Japan."],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    print("SentenceTransformer loaded OK.")
    print("Embedding shape:", vec.shape)

except Exception as e:
    print("SentenceTransformer path failed:", repr(e))
    print("Trying plain transformers fallback...")

    emb_tok = AutoTokenizer.from_pretrained(EMBED_MODEL_NAME, use_fast=True, **hf_kwargs)
    if emb_tok.pad_token is None:
        emb_tok.pad_token = emb_tok.eos_token

    emb_model = AutoModel.from_pretrained(
        EMBED_MODEL_NAME,
        dtype=DTYPE,
        **hf_kwargs,
    )
    if DEVICE != "cuda":
        emb_model = emb_model.to(DEVICE)
    emb_model.eval()

    def mean_pool(last_hidden_state, attention_mask):
        mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
        summed = (last_hidden_state * mask).sum(dim=1)
        denom = mask.sum(dim=1).clamp(min=1e-8)
        return summed / denom

    toks = emb_tok(
        ["What is the capital of Japan?", "Tokyo is the capital of Japan."],
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=256,
    )
    toks = {k: v.to(DEVICE) for k, v in toks.items()}

    with torch.no_grad():
        res = emb_model(**toks)
        pooled = mean_pool(res.last_hidden_state, toks["attention_mask"])
        pooled = torch.nn.functional.normalize(pooled, p=2, dim=-1)

    print("Fallback embedding model loaded OK.")
    print("Embedding shape:", tuple(pooled.shape))


Loading sentence-transformers embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


SentenceTransformer loaded OK.
Embedding shape: (2, 384)



## Pass criteria

You are ready to rerun the main harness if these are true:
- `cuda available: True`
- GPU name shows your 4060
- generation model loads without exception
- embedding model loads without exception
- tiny generation test returns output

Once that is true, go back to the main harness notebook and set:

```python
LOCAL_FILES_ONLY = True
```

after the first successful cache build.
